# Evaluación RAG — Fase 1: Recuperación (Retrieval)

Este notebook evalúa la fase de **recuperación** de tu pipeline RAG financiero contra tu índice de **Pinecone**, usando el `golden_dataset.json` (63 consultas, generado a partir de `noticias_enriquecido.json` y `noticias_nodes.json` reales).

Métricas implementadas:
- **Precision@K**
- **Recall@K**
- **F1-Score** (a partir de Precision/Recall)
- **MRR** (Mean Reciprocal Rank)
- **NDCG@K** (con relevancia graduada 0–3)

Se evalúa para **K = 5 y K = 10**.

### Diseño de relevancia (0–3), sobre metadatos de cada nodo recuperado
| Grado | Condición |
|---|---|
| **3** | mismo `ticker` **y** `fecha` dentro del rango del `periodo` objetivo (el evento exacto) |
| **2** | mismo `indice_sector`, pero `periodo` distinto (mismo sector, otro episodio) |
| **1** | mismo `periodo`, pero `indice_sector` distinto (mismo episodio macro, otro sector — solo aplica en 2 de los 63 periodos que cruzan sector) |
| **0** | cualquier otro caso |

El conjunto **binario relevante** (para Precision/Recall/F1/MRR) es el de **grado 3**.

> ⚠️ Este notebook necesita: tu **API key de Pinecone**, el **nombre de tu índice**, y el **mismo modelo de embeddings** que usaste para indexar (por defecto se asume `BAAI/bge-m3`, ajusta si usaste otro).

In [ ]:
# @title 1. Instalación de dependencias
!pip install -q pinecone sentence-transformers pandas matplotlib tqdm

## 2. Subir archivos necesarios

Sube estos dos archivos (botón de carpeta a la izquierda en Colab, o usa `files.upload()` abajo):
- `golden_dataset.json`
- `noticias_nodes__1_.json` (opcional, solo se usa para inspección/estadísticas; el ground truth ya está resuelto dentro de `golden_dataset.json`)

In [ ]:
from google.colab import files
import os

uploaded = files.upload()  # selecciona golden_dataset.json (y opcionalmente noticias_nodes__1_.json)
print("Archivos subidos:", list(uploaded.keys()))

## 3. Configuración

Rellena tu API key de Pinecone y el nombre del índice. Ajusta `EMBEDDING_MODEL` si no usaste BGE-M3 para indexar (debe ser **exactamente** el mismo modelo que usaste al crear el índice, o las similitudes no serán comparables).

In [ ]:
# @title Configuración
try:
    from google.colab import userdata
    PINECONE_API_KEY = userdata.get("PINECONE_API_KEY")
except ImportError:
    import getpass
    PINECONE_API_KEY = getpass.getpass("PINECONE_API_KEY: ")
PINECONE_INDEX_NAME = "noticias-financieras"  # @param {type:"string"}
PINECONE_NAMESPACE = ""  # @param {type:"string"}  # deja vacío si no usas namespaces

EMBEDDING_MODEL = "BAAI/bge-m3"  # @param {type:"string"}

K_VALUES = [5, 10]  # valores de K solicitados
TOP_K_MAX = max(K_VALUES)

GOLDEN_PATH = "golden_dataset.json"

assert PINECONE_API_KEY, "Falta PINECONE_API_KEY"
assert PINECONE_INDEX_NAME, "Falta PINECONE_INDEX_NAME"


In [ ]:
# @title 4. Cargar golden dataset
import json

with open(GOLDEN_PATH, encoding="utf-8") as f:
    golden = json.load(f)

print(f"Consultas en el golden dataset: {len(golden)}")
golden[0]

## 5. Conexión a Pinecone y carga del modelo de embeddings

In [ ]:
from pinecone import Pinecone
from sentence_transformers import SentenceTransformer

pc = Pinecone(api_key=PINECONE_API_KEY)
index = pc.Index(PINECONE_INDEX_NAME)
print(index.describe_index_stats())

model = SentenceTransformer(EMBEDDING_MODEL)

def embed_query(text: str):
    """Genera el embedding de una query con el mismo modelo usado para indexar.
    Ajusta normalize_embeddings/instrucciones si tu pipeline de indexación las usaba."""
    return model.encode(text, normalize_embeddings=True).tolist()

## 6. Función de recuperación (retrieval)

Devuelve, para una query, la lista de metadatos de los `TOP_K_MAX` nodos recuperados de Pinecone, en orden de ranking. Ajusta los nombres de campo en `metadata` si tu índice los guardó con otras claves.

In [ ]:
def retrieve(query: str, top_k: int = TOP_K_MAX):
    vector = embed_query(query)
    kwargs = dict(vector=vector, top_k=top_k, include_metadata=True)
    if PINECONE_NAMESPACE:
        kwargs["namespace"] = PINECONE_NAMESPACE
    res = index.query(**kwargs)
    hits = []
    for match in res["matches"]:
        md_ = match.get("metadata", {}) or {}
        hits.append({
            "id": match["id"],
            "score": match["score"],
            "ticker": md_.get("ticker"),
            "indice_sector": md_.get("indice_sector"),
            "periodo": md_.get("periodo"),
            "fecha": (md_.get("fecha") or "")[:10],
            "url": md_.get("url"),
        })
    return hits

## 7. Asignación de grado de relevancia (0–3)

Aplica las reglas del enunciado sobre los metadatos de cada nodo recuperado, comparados contra el `target_ticker` / `target_sector` / `target_periodo` / `date_range` de la consulta del golden dataset.

In [ ]:
def grade_relevance(golden_entry: dict, hit: dict) -> int:
    ticker_match = hit["ticker"] == golden_entry["target_ticker"]
    sector_match = hit["indice_sector"] == golden_entry["target_sector"]
    periodo_match = hit["periodo"] == golden_entry["target_periodo"]
    date_start, date_end = golden_entry["date_range"]
    date_in_range = bool(hit["fecha"]) and (date_start <= hit["fecha"] <= date_end)

    if ticker_match and periodo_match and date_in_range:
        return 3
    if sector_match and not periodo_match:
        return 2
    if periodo_match and not sector_match:
        return 1
    return 0

## 8. Métricas de recuperación

- **Precision@K** = relevantes (grado 3) en el top K / K
- **Recall@K** = relevantes (grado 3) en el top K / total de relevantes (grado 3) en todo el corpus, `n_relevant_nodes_grade3` del golden dataset
- **F1@K** = media armónica de Precision@K y Recall@K
- **MRR** = 1 / (posición del primer resultado de grado 3), calculado sobre el ranking completo recuperado (hasta `TOP_K_MAX`); 0 si no aparece ninguno
- **NDCG@K** = DCG@K / IDCG@K, usando la relevancia graduada (0–3) de cada hit. El IDCG@K se calcula con el ranking ideal REAL (a partir de los conteos de grado 3/2/1 del golden dataset), no reordenando lo recuperado — así el NDCG cae a ~0 cuando el sistema falla por completo, en vez de quedar artificialmente alto.

In [ ]:
import math

def precision_at_k(grades, k):
    topk = grades[:k]
    return sum(1 for g in topk if g == 3) / k

def recall_at_k(grades, k, total_relevant):
    if total_relevant == 0:
        return None  # no debería pasar: el golden dataset garantiza >=1 nodo grado 3 por query
    topk = grades[:k]
    return sum(1 for g in topk if g == 3) / total_relevant

def f1_score(precision, recall):
    if precision is None or recall is None or (precision + recall) == 0:
        return 0.0
    return 2 * precision * recall / (precision + recall)

def reciprocal_rank(grades):
    for i, g in enumerate(grades, start=1):
        if g == 3:
            return 1.0 / i
    return 0.0

def dcg_at_k(grades, k):
    return sum(g / math.log2(i + 1) for i, g in enumerate(grades[:k], start=1))

def ideal_grades_at_k(entry, k):
    """Construye el ranking IDEAL de tamaño k a partir de los conteos REALES
    de relevancia en todo el corpus (golden_dataset.json), no a partir de lo
    que el sistema recuperó. Llena primero con grado 3 (hasta agotar el total
    disponible), luego grado 2, luego grado 1, y rellena el resto con 0.
    Esto corrige un bug de la versión anterior: reordenar únicamente los k
    resultados recuperados como si fueran el 'ideal' infla el NDCG cuando el
    sistema falla por completo (ninguna de las k recuperaciones es relevante),
    porque el IDCG resultante también cae a 0 y el cociente deja de reflejar
    el fallo real."""
    counts = {
        3: entry["n_relevant_nodes_grade3"],
        2: entry["n_relevant_nodes_grade2_only_sector"],
        1: entry["n_relevant_nodes_grade1_only_periodo"],
    }
    ideal = []
    for grade in (3, 2, 1):
        if len(ideal) >= k:
            break
        take = min(counts[grade], k - len(ideal))
        ideal.extend([grade] * take)
    while len(ideal) < k:
        ideal.append(0)
    return ideal


def ndcg_at_k(grades, entry, k):
    dcg = dcg_at_k(grades, k)
    ideal = ideal_grades_at_k(entry, k)
    idcg = dcg_at_k(ideal, k)
    if idcg == 0:
        return 0.0
    return dcg / idcg

## 9. Bucle de evaluación

Ejecuta cada consulta del golden dataset contra Pinecone, calcula grados de relevancia y todas las métricas para K=5 y K=10.

In [ ]:
from tqdm.auto import tqdm
import pandas as pd

rows = []
per_query_hits = {}  # útil para depurar manualmente una query concreta

for entry in tqdm(golden, desc="Evaluando consultas"):
    hits = retrieve(entry["query"], top_k=TOP_K_MAX)
    grades = [grade_relevance(entry, h) for h in hits]
    per_query_hits[entry["query_id"]] = list(zip(hits, grades))

    total_relevant = entry["n_relevant_nodes_grade3"]
    mrr = reciprocal_rank(grades)

    row = {
        "query_id": entry["query_id"],
        "query": entry["query"],
        "target_ticker": entry["target_ticker"],
        "target_sector": entry["target_sector"],
        "target_periodo": entry["target_periodo"],
        "n_retrieved": len(hits),
        "n_relevant_total_grade3": total_relevant,
        "mrr": mrr,
    }

    for k in K_VALUES:
        p = precision_at_k(grades, k)
        r = recall_at_k(grades, k, total_relevant)
        row[f"precision@{k}"] = p
        row[f"recall@{k}"] = r
        row[f"f1@{k}"] = f1_score(p, r)
        row[f"ndcg@{k}"] = ndcg_at_k(grades, entry, k)

    rows.append(row)

results_df = pd.DataFrame(rows)
results_df.head()

## 10. Resultados agregados

Media de cada métrica sobre las 63 consultas, para K=5 y K=10, más MRR global.

In [ ]:
summary_cols = ["mrr"]
for k in K_VALUES:
    summary_cols += [f"precision@{k}", f"recall@{k}", f"f1@{k}", f"ndcg@{k}"]

summary = results_df[summary_cols].mean().to_frame("media").round(4)
summary

## 11. Visualización comparativa (K=5 vs K=10)

In [ ]:
import matplotlib.pyplot as plt

metrics = ["precision", "recall", "f1", "ndcg"]
x = range(len(metrics))
width = 0.35

fig, ax = plt.subplots(figsize=(8, 5))
for i, k in enumerate(K_VALUES):
    values = [results_df[f"{m}@{k}"].mean() for m in metrics]
    ax.bar([xi + i * width for xi in x], values, width, label=f"K={k}")

ax.axhline(results_df["mrr"].mean(), color="red", linestyle="--", label=f"MRR = {results_df['mrr'].mean():.3f}")
ax.set_xticks([xi + width / 2 for xi in x])
ax.set_xticklabels([m.capitalize() for m in metrics])
ax.set_ylabel("Valor medio")
ax.set_title("Métricas de recuperación — Golden Dataset (63 consultas)")
ax.legend()
plt.tight_layout()
plt.savefig("metricas_retrieval.png", dpi=150)
plt.show()

## 12. Peores consultas (para depurar el índice / el threshold)

Ordena por `f1@10` ascendente: las consultas donde el sistema recupera peor son las primeras candidatas a revisar (¿threshold de similitud demasiado alto? ¿chunking demasiado fino? ¿metadatos mal indexados?).

In [ ]:
worst = results_df.sort_values("f1@10").head(10)[
    ["query_id", "query", "target_ticker", "target_periodo", "precision@10", "recall@10", "f1@10", "ndcg@10", "mrr"]
]
worst

## 13. Guardar resultados

In [ ]:
results_df.to_csv("resultados_retrieval_detalle.csv", index=False)
summary.to_csv("resultados_retrieval_resumen.csv")

from google.colab import files as colab_files
colab_files.download("resultados_retrieval_detalle.csv")
colab_files.download("resultados_retrieval_resumen.csv")
colab_files.download("metricas_retrieval.png")

## Apéndice — Inspeccionar una consulta concreta

Útil para depurar manualmente: ver los hits reales de Pinecone y el grado de relevancia asignado a cada uno.

In [ ]:
query_id_to_inspect = "q001"  # @param {type:"string"}

for hit, grade in per_query_hits[query_id_to_inspect]:
    print(f"[grade={grade}] score={hit['score']:.3f} ticker={hit['ticker']} periodo={hit['periodo']} fecha={hit['fecha']} url={hit['url']}")